In [40]:
from collections.abc import Generator
from pathlib import Path
from typing import Any

import datasets
import pandas as pd
from datasets import Features, Sequence, Value

# 1. Prepare a univariate dataset for pre-training/fine-tuning
In this example, we will see how to use the Hugging Face ```datasets``` library to prepare your custom datasets to use with ```uni2ts```. 

Firstly, we load our data which comes in the form of a wide dataframe. Here, each column represents a _univariate_ time series.

In [41]:
# Load dataframe
url_wide = (
    "https://gist.githubusercontent.com/rsnirwan/c8c8654a98350fadd229b00167174ec4"
    "/raw/a42101c7786d4bc7695228a0f2c8cea41340e18f/ts_wide.csv"
)
df = pd.read_csv(url_wide, index_col=0, parse_dates=True)

df.head()

,A,B,C,D,E,F,G,H,I,J
2021-01-01 00:00:00,-1.3378,0.1268,-0.3645,-1.0864,-2.3803,-0.2447,2.2647,-0.7917,0.7071,1.3763
2021-01-01 01:00:00,-1.6111,0.0926,-0.1364,-1.1613,-2.1421,-0.3477,2.4262,-0.9609,0.6413,1.2750
2021-01-01 02:00:00,-1.9259,-0.1420,0.1063,-1.0405,-2.1426,-0.3271,2.4434,-0.9034,0.4323,0.6767
2021-01-01 03:00:00,-1.9184,-0.4930,0.6269,-0.8531,-1.7060,-0.3088,2.4307,-0.9602,0.3193,0.5150
2021-01-01 04:00:00,-1.9168,-0.5057,0.9419,-0.7666,-1.4287,-0.4284,2.3258,-1.2504,0.3660,0.1708


In [42]:
print(df.shape)

(240, 10)


### Method 1: Example generator function
1. Create an example generator function, a function which yields each individual time series. Each time series consists of 
    1. target: target time series that should be predicted
    2. start: timestamp of the first time step
    3. freq: frequency str of time series
    4. item_id: identifier 
    5. (optional) past_feat_dynamic_real: time series for which only the context values are known
    6. (optional) feat_dynamic_real: time series for which the context and prediction values are known
2. Define the schema for the features to ensure the datasets library saves the correct data types.
3. Write the data to disk using the ```from_generator``` function.

In [43]:
# Defines a generator function named 'example_gen_func' that will yield a dictionary.
# The function returns a generator object, which is a special type of iterator in Python.
def example_gen_func() -> Generator[dict[str, Any]]:

    # Loops over the range of the number of columns in the DataFrame 'df'.
    # 'df.columns' refers to the column names of the DataFrame, and 'len(df.columns)' returns the total number of columns.
    for i in range(len(df.columns)):

        # The 'yield' keyword is used here to produce a dictionary for each column in 'df' one by one.
        # The dictionary contains the following key-value pairs:
        yield {
            # "target": The target time series values for the current column, converted to a NumPy array.
            # 'df.iloc[:, i]' selects all rows from the i-th column of 'df', and 'to_numpy()' converts it into a NumPy array.
            "target": df.iloc[:, i].to_numpy(),  # array of shape (time,)

            # "start": The start time of the time series, which is the first index of the DataFrame.
            # 'df.index[0]' gives the first timestamp in the DataFrame's index.
            "start": df.index[0],

            # "freq": The frequency of the time series. 'pd.infer_freq(df.index)' infers the frequency (like daily, monthly, etc.) from the DataFrame's index.
            "freq": pd.infer_freq(df.index),

            # "item_id": A unique identifier for the time series, in this case using a string formatted with the current column index 'i'.
            # The item_id could be useful to distinguish between multiple time series.
            "item_id": f"item_{i}",
        }

In [44]:
# Defines a 'Features' object that is a structured representation of the data fields and their types.
# This object helps specify the expected types of various features in your dataset.

features = Features(

    # 'dict()' is used to map feature names (keys) to their corresponding data types (values).
    dict(

        # 'target': The target field, which represents the actual time series values (e.g., demand, stock prices).
        # It is represented as a sequence of float32 values, where each element in the sequence is a floating-point number.
        target=Sequence(Value("float32")),

        # 'start': The start time of the time series, stored as a timestamp in seconds.
        # The 'Value("timestamp[s]")' defines that this field should hold a timestamp value in seconds precision.
        start=Value("timestamp[s]"),

        # 'freq': The frequency of the time series (e.g., daily, hourly), represented as a string.
        # This field describes the time intervals between data points.
        freq=Value("string"),

        # 'item_id': A unique identifier for the time series, also represented as a string.
        # This field helps distinguish between different time series in the dataset.
        item_id=Value("string"),
    )
)


In [45]:
# Creates a Hugging Face dataset (hf_dataset) from a generator function.
# 'datasets.Dataset.from_generator' takes the generator function 'example_gen_func' and uses it to populate the dataset.
# 'features=features' ensures that the structure and data types (target, start, freq, item_id) are applied to the dataset as specified earlier.
hf_dataset = datasets.Dataset.from_generator(example_gen_func, features=features)

# Saves the Hugging Face dataset to disk at the specified location, in this case, "example_dataset_1".
# 'Path("example_dataset_1")' defines the folder where the dataset will be stored.
hf_dataset.save_to_disk(Path("example_dataset_1"))


Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

### Method 2: Sharded example generator function
For larger datasets, the Hugging Face ```datasets``` library is able to use multiprocessing to speed up the generation of examples. Since the ```from_generator``` function takes as input a generator object which iterates through every example, naively using this function with multiprocessing does not lead to any speed ups. Instead, we need to provide a _sharded_ generator function, which is able to index into the specific examples based on the inputs. See the following example for a simple recipe:

In [25]:
def sharded_example_gen_func(examples: list[int]) -> Generator[dict[str, Any]]:
    for i in examples:
        yield {
            "target": df.iloc[:, i].to_numpy(),
            "start": df.index[0],
            "freq": pd.infer_freq(df.index),
            "item_id": f"item_{i}",
        }

In [26]:
features = Features(
    dict(
        target=Sequence(Value("float32")),
        start=Value("timestamp[s]"),
        freq=Value("string"),
        item_id=Value("string"),
    )
)

In [46]:
hf_dataset = datasets.Dataset.from_generator(
    sharded_example_gen_func,
    features=features,
    gen_kwargs={"examples": [i for i in range(len(df.columns))]},
    num_proc=2,
)
hf_dataset.save_to_disk(Path("example_dataset_2"))

Generating train split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

# 2. Prepare a multivariate dataset for pre-training/fine-tuning
Finally, we can also prepare _multivariate_ time series:

In [28]:
# Load dataframe
url_wide = (
    "https://gist.githubusercontent.com/rsnirwan/c8c8654a98350fadd229b00167174ec4"
    "/raw/a42101c7786d4bc7695228a0f2c8cea41340e18f/ts_wide.csv"
)
df = pd.read_csv(url_wide, index_col=0, parse_dates=True)

df.head()

,A,B,C,D,E,F,G,H,I,J
2021-01-01 00:00:00,-1.3378,0.1268,-0.3645,-1.0864,-2.3803,-0.2447,2.2647,-0.7917,0.7071,1.3763
2021-01-01 01:00:00,-1.6111,0.0926,-0.1364,-1.1613,-2.1421,-0.3477,2.4262,-0.9609,0.6413,1.2750
2021-01-01 02:00:00,-1.9259,-0.1420,0.1063,-1.0405,-2.1426,-0.3271,2.4434,-0.9034,0.4323,0.6767
2021-01-01 03:00:00,-1.9184,-0.4930,0.6269,-0.8531,-1.7060,-0.3088,2.4307,-0.9602,0.3193,0.5150
2021-01-01 04:00:00,-1.9168,-0.5057,0.9419,-0.7666,-1.4287,-0.4284,2.3258,-1.2504,0.3660,0.1708


In [11]:
# Defines a generator function named 'multivar_example_gen_func' that will yield a dictionary.
# The function returns a generator object, which is a special type of iterator in Python.
# The generator produces a dictionary with the structure expected for a multivariate time series.
def multivar_example_gen_func() -> Generator[dict[str, Any], None, None]:

    # The 'yield' keyword is used to return the dictionary. In this case, it produces a multivariate time series.
    yield {
        # 'target': The target time series values for all variables. 'df.to_numpy().T' converts the DataFrame into a NumPy array 
        # and transposes it to have a shape of (var, time), where 'var' is the number of variables and 'time' is the number of time steps.
        "target": df.to_numpy().T,  # array of shape (var, time) -> (10, 240)

        # 'start': The start time of the time series, which is the first index of the DataFrame.
        "start": df.index[0],

        # 'freq': The frequency of the time series, inferred from the DataFrame's index using pandas' 'infer_freq' method.
        "freq": pd.infer_freq(df.index),

        # 'item_id': A unique identifier for this multivariate time series. Since this generator creates only one time series,
        # 'item_id' is set to "item_0".
        "item_id": "item_0",
    }


In [12]:
features = Features(
    dict(
        target=Sequence(
            Sequence(Value("float32")), length=len(df.columns)
        ),  # multivariate time series are saved as (var, time)
        start=Value("timestamp[s]"),
        freq=Value("string"),
        item_id=Value("string"),
    )
)

In [13]:
hf_dataset = datasets.Dataset.from_generator(
    multivar_example_gen_func, features=features
)
hf_dataset.save_to_disk("example_dataset_multi")

Generating train split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

# 3. Inspecting the processed data
Let's inspect the processed datasets to ensure that our data has been processed correctly.

In [30]:
# Load datasets with ArrowTableIndexer
ds1 = datasets.load_from_disk("example_dataset_1").with_format("numpy")
ds2 = datasets.load_from_disk("example_dataset_2").with_format("numpy")
ds_multi = datasets.load_from_disk("example_dataset_multi").with_format("numpy")

```example_dataset_1``` and ```example_dataset_2``` are univariate datasets, which should have 10 time series each, and ```example_dataset_multi``` should be a single multivariate time series (with 10 variates). 

In [15]:
len(ds1), len(ds2), len(ds_multi)

(10, 10, 1)

Inspecting the features returned when we index into a time series from the dataset...

In [33]:
ds1.shape, ds2.shape, ds_multi.shape

((10, 4), (10, 4), (1, 4))

In [16]:
ds1[0].keys(), ds2[0].keys(), ds_multi[0].keys()

(dict_keys(['target', 'start', 'freq', 'item_id']),
 dict_keys(['target', 'start', 'freq', 'item_id']),
 dict_keys(['target', 'start', 'freq', 'item_id']))

We should get 2 univariate and 1 multivariate target time series...

In [17]:
ds1[0]["target"].shape, ds2[0]["target"].shape, ds_multi[0]["target"].shape

((240,), (240,), (10, 240))